# N-gram model

Count-based n-gram over `data/processed/names.txt` (no training: the model is
the smoothed count tensor). The logic lives in the package — `onomatron.data`
(names, vocab, n-gram pairs) and `onomatron.ngram.NGramModel` — this notebook
only wires it up, plots the bigram count matrix and samples names. n=2 is the
plain bigram model.


In [ ]:
import matplotlib.pyplot as plt
import torch

%matplotlib inline

from onomatron.data import Vocab, load_names
from onomatron.ngram import NGramModel

## Load names and build the vocab

The `'.'` sentinel is index 0 and marks both the beginning and the end of a
name — safe because the pipeline guarantees names.txt holds only latin letters
and spaces.


In [ ]:
names = load_names("../data/processed/names.txt")
vocab = Vocab(names)
len(names), len(vocab)

## Fit the bigram model (n=2)


In [ ]:
model = NGramModel.fit(names, vocab, n=2)
model.counts.shape

## Count matrix (bigram, n=2)


In [ ]:
N = model.counts
plt.figure(figsize=(24, 24))
plt.imshow(N, cmap="Blues")
for i in range(len(vocab)):
    for j in range(len(vocab)):
        chstr = vocab.itos[i] + vocab.itos[j]
        plt.text(j, i, chstr, ha="center", va="bottom", color="gray")
        plt.text(j, i, N[i, j].item(), ha="center", va="top", color="gray")
plt.axis("off")

## Sample names (n=2)


In [ ]:
for name in model.sample(n=10, seed=135798):
    print(name)

## In-sample evaluation


In [ ]:
print(f"mean NLL per bigram (in-sample): {model.nll(names):.4f}")

## Fit / predict / evaluate — train/val split, orders compared

Shuffle, hold out 10%, fit on the train split only, then evaluate on both.
Watch the val NLL as n grows: past n=3 the count tensor gets too sparse and
the smoothing floor dominates.


In [ ]:
g = torch.Generator().manual_seed(42)
perm = torch.randperm(len(names), generator=g).tolist()
split = int(0.9 * len(names))
train = [names[i] for i in perm[:split]]
val = [names[i] for i in perm[split:]]
print(f"{len(train)} train / {len(val)} val names")

for n in (2, 3, 4):
    m = NGramModel.fit(train, vocab, n=n)
    print(f"n={n}: train NLL {m.nll(train):.4f} | val NLL {m.nll(val):.4f}")

m3 = NGramModel.fit(train, vocab, n=3)
print("samples (n=3):")
for name in m3.sample(n=5, seed=135798):
    print(name)